# iTransformer backtesting — strategy tuning and out-of-sample evaluation

The shared backtesting pipeline, with the iTransformer forecasts from
`01_itransformer_training.ipynb` as the signal:

1. **Three strategies**, imported from `backtesting/backtest_functions.py` so
   that both model families are evaluated by literally the same code —
   simple threshold, volatility-adjusted (risk-based) and momentum.
2. **Parameters tuned on the training period only**, by 10-fold chronological
   cross-validation with an Optuna study per fold maximising the annualised
   Sharpe ratio of the fold's portfolio path, then averaged across folds.
3. **Evaluation on two held-out periods** — uptrend Jun–Dec 2024 and sideways
   Jan–Jul 2025, each losing its first 168 hours to sequence construction, which
   gives the 2024-06-08 → 2024-11-30 and 2025-01-08 → 2025-06-30 evaluation
   index — with a 0.0275 % per-trade transaction cost and $10,000 of initial
   capital.

The backtesting stage departs from notebook 01's forecasting protocol in two
deliberate ways, both shared with the LSTM backtesting run so the two studies'
backtesting tables line up row by row:

* **Sentiment source.** The forecaster is trained on the `_full` sentiment
  column and backtested on the `_ps2` one (see `itransformer_common.py`). Both are
  scorings of the same headline stream and correlate at 0.998 on shared
  timestamps; the second carries about 3,000 more headlines and runs to Nov 2025
  rather than Mar 2025, which is what makes the full Jan–Jul 2025 sideways
  window available at all.
* **Evaluation windows.** Periods and the strategy-search cut-off come from
  `create_validation_periods()` in `backtesting/data_loader.py`, not from the
  forecasting split. The models were fitted on data before 2024-04-30, so both
  periods remain fully out-of-sample.

The signal is the seed-averaged forecast of the three iTransformers trained per feature
set, which is the same object the Diebold–Mariano tests in notebook 01 use.

Trade logs are written to `backtest_results/<feature_set>/` under the same file
names as the iTransformer runs, so the two studies can be compared
folder-by-folder.

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import itransformer_common as L
from backtest_functions import (backtest_profit_threshold_with_trades,
                                backtest_profit_threshold,
                                backtest_ts_momentum)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ------------------------------------------------------------------ toggles
N_SPLITS = 10
N_TRIALS_THRESHOLD = 200
N_TRIALS_RISK = 500
N_TRIALS_MOMENTUM = 1000
N_JOBS = 8
INITIAL_CAPITAL = L.INITIAL_CAPITAL
COST_PCT = L.TRANSACTION_COST_PCT
RUN_CV = True            # False -> reuse out/itransformer_strategy_params.json

print(f"initial capital      : ${INITIAL_CAPITAL:,.0f}")
print(f"transaction cost     : {COST_PCT}% per trade")
print(f"CV folds / Optuna    : {N_SPLITS} folds, "
      f"{N_TRIALS_THRESHOLD}/{N_TRIALS_RISK}/{N_TRIALS_MOMENTUM} trials")
print(f"sentiment (backtest) : {L.SENT_FILE_BACKTEST}")
print(f"strategy search on   : data before {L.BT_TRAIN_END.date()}")
for _k, _p in L.BT_PERIODS.items():
    print(f"period {_k} ({_p['short']:8s}): {_p['start'].date()} -> {_p['end'].date()}")

# ------------------------------------------------- model-free momentum controls
# Model-free momentum controls. The momentum rule's signal is
#     (price[i-1] + pred[i] - price[i-lookback]) / price[i-lookback]
# so the forecaster contributes only the single term pred[i] -- tens of dollars
# against a lookback move of thousands. Re-tuning and re-running the *identical*
# rule with pred = 0 gives pure time-series momentum with no model at all; with
# pred permuted in time it gives a random-forecast control that keeps the
# forecast distribution and destroys only its timing. Whatever the forecaster
# genuinely contributes has to show up as a gap between these rows and the model
# rows; if it does not, the returns belong to the momentum filter.
BENCHMARKS = {"no_model_zero": "zero", "no_model_shuffled": "shuffled"}
BENCH_LABEL = {"no_model_zero": "Pure time-series momentum (no forecast)",
               "no_model_shuffled": "Momentum with time-shuffled forecast"}


## 1. Data and trained models

In [ ]:
HOURLY = L.build_feature_sets(L.SENT_FILE_BACKTEST)
print()
for n in L.FEATURE_SET_ORDER:
    d = HOURLY[n]
    print(f"{n:26s} rows={len(d):,}  {d.index.min()} -> {d.index.max()}")

PREDICTORS = {n: L.ITransformerPredictor.load(L.CKPT_DIR / f"itransformer_{L.FOLDER_NAME[n]}.pt")
              for n in L.FEATURE_SET_ORDER}
print()
for n, p in PREDICTORS.items():
    # the backtest frame must present the variates in the order the model was
    # trained on, or the stored scaler and the input channels would misalign
    assert list(HOURLY[n].columns) == p.columns, (
        f"{n}: column order differs between training and backtest frames")
    print(f"{n:26s} variates={p.num_variates:2d}  seeds={len(p.models)}  hp={p.hp}")

REGIMES = L.BT_PERIODS

## 2. Signals

One forward pass per feature set over the training period (used only for
parameter search) and over each test regime.

In [ ]:
SIGNALS = {}
for name in L.FEATURE_SET_ORDER:
    dfh = HOURLY[name]
    for r, info in REGIMES.items():
        X, y, idx, close_prev, close = L.period_windows(dfh, info["start"], info["end"])
        SIGNALS[(name, r)] = dict(pred=PREDICTORS[name].predict_windows(X), y=y,
                                  idx=idx, close_prev=close_prev, close=close)

sig_tbl = pd.DataFrame([
    dict(feature_set=n, regime=REGIMES[r]["short"], n=len(SIGNALS[(n, r)]["pred"]),
         start=SIGNALS[(n, r)]["idx"].min(), end=SIGNALS[(n, r)]["idx"].max(),
         pred_mean=SIGNALS[(n, r)]["pred"].mean(),
         pred_std=SIGNALS[(n, r)]["pred"].std(),
         realised_std=SIGNALS[(n, r)]["y"].std())
    for n in L.FEATURE_SET_ORDER for r in REGIMES])
print(sig_tbl.to_string(index=False))

## 3. Strategy parameter search (10-fold CV + Optuna, training period only)

The search spaces are the ones documented for the iTransformer study:

* threshold — $\tau \in [0.01, 2.0]$ %, hold $H \in [1, 168]$ h;
* volatility-adjusted — $\tau \in [0.01, 2.0]$ %, per-trade risk $\in [0.01, 2.5]$ %, $H \in [1, 168]$ h;
* momentum — lookback $L \in [1, 168]$ h, $H \in [1, 168]$ h, signal threshold $\in [0.01, 2.0]$ %, risk $\in [0, 2.5]$ %.

In [ ]:
PARAM_PATH = L.OUT / "itransformer_strategy_params.json"

if RUN_CV:
    PARAMS, FOLDS = {}, {}
    t_all = time.time()
    for name in L.FEATURE_SET_ORDER:
        train_frame = HOURLY[name].loc[HOURLY[name].index < L.BT_TRAIN_END]
        print(f"\n{'=' * 70}\n{L.NICE_NAME[name]}  ({len(train_frame):,} training hours)\n{'=' * 70}")

        print("threshold strategy")
        p, f = L.cv_optimize_threshold(train_frame, PREDICTORS[name], n_splits=N_SPLITS,
                                       n_trials=N_TRIALS_THRESHOLD, n_jobs=N_JOBS)
        PARAMS[(name, "threshold")], FOLDS[(name, "threshold")] = p, f

        print("volatility-adjusted strategy")
        p, f = L.cv_optimize_risk(train_frame, PREDICTORS[name], n_splits=N_SPLITS,
                                  n_trials=N_TRIALS_RISK, n_jobs=N_JOBS)
        PARAMS[(name, "risk_based")], FOLDS[(name, "risk_based")] = p, f

        print("momentum strategy")
        p, f = L.cv_optimize_momentum(train_frame, PREDICTORS[name], n_splits=N_SPLITS,
                                      n_trials=N_TRIALS_MOMENTUM, n_jobs=N_JOBS)
        PARAMS[(name, "momentum")], FOLDS[(name, "momentum")] = p, f

    print(f"\ntotal parameter-search time: {(time.time() - t_all) / 60:.1f} min")
    PARAM_PATH.write_text(json.dumps(
        {f"{n}|{s}": PARAMS[(n, s)] for (n, s) in PARAMS}, indent=2))
    pd.concat([f.assign(feature_set=n, strategy=s) for (n, s), f in FOLDS.items()]) \
        .to_csv(L.OUT / "itransformer_strategy_cv_folds.csv", index=False)
else:
    blob = json.loads(PARAM_PATH.read_text())
    PARAMS = {tuple(k.split("|")): v for k, v in blob.items()}
    print("reusing", PARAM_PATH)

param_tbl = pd.DataFrame([
    dict(feature_set=n, strategy=s, **PARAMS[(n, s)])
    for n in L.FEATURE_SET_ORDER
    for s in ("threshold", "risk_based", "momentum")])
print()
print(param_tbl.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# ---------------------------------------------------- model-free momentum search
# Same rule, same folds, same trial budget, same aggregate-Sharpe selection.
if RUN_CV:
    _pv = HOURLY["price+volume"]
    _pv_train = _pv.loc[_pv.index < L.BT_TRAIN_END]
    for _key, _mode in BENCHMARKS.items():
        print(f"\n{'=' * 70}\n{BENCH_LABEL[_key]}  ({len(_pv_train):,} training hours)\n{'=' * 70}")
        p, f = L.cv_optimize_momentum(_pv_train, PREDICTORS["price+volume"],
                                      n_splits=N_SPLITS, n_trials=N_TRIALS_MOMENTUM,
                                      n_jobs=N_JOBS, forecast=_mode)
        PARAMS[(_key, "momentum")], FOLDS[(_key, "momentum")] = p, f

    PARAM_PATH.write_text(json.dumps(
        {f"{n}|{s}": PARAMS[(n, s)] for (n, s) in PARAMS}, indent=2))
    pd.concat([f.assign(feature_set=n, strategy=s) for (n, s), f in FOLDS.items()]) \
        .to_csv(L.OUT / "itransformer_strategy_cv_folds.csv", index=False)


## 4. Backtests on the held-out regimes

Each strategy is run with its CV-selected parameters on both regimes; the trade
log is written to `backtest_results/<feature_set>/`.

In [ ]:
STRATEGY_LABEL = {"threshold": "Threshold", "risk_based": "Risk-Based",
                  "momentum": "Momentum"}
FILE_PREFIX = {"threshold": "threshold_backtest_trades",
               "risk_based": "risk_based_backtest_trades",
               "momentum": "momentum_backtest_trades"}

COLORS = {"long": "green", "short": "red",
          "exit_long": "lightgreen", "exit_short": "lightcoral",
          "start": "grey", "end": "black"}
MARKERS = {"long": "^", "short": "v", "exit_long": "o", "exit_short": "o",
           "start": "s", "end": "s"}


def plot_trades(trades_df, title):
    plt.figure(figsize=(12, 5))
    for tt in trades_df["trade_type"].unique():
        sub = trades_df[trades_df["trade_type"] == tt]
        plt.scatter(sub["datetime"], sub["portfolio_value"],
                    label=f'{tt.replace("_", " ").title()} ({len(sub)})',
                    marker=MARKERS.get(tt, "o"),
                    s=50 if tt in ("long", "short") else 25,
                    alpha=0.8, color=COLORS.get(tt, "blue"))
    plt.xlabel("Time")
    plt.ylabel("Portfolio Value at Trade ($)")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def run_backtest(strategy, prices, preds, params):
    if strategy == "threshold":
        pv, tt, tv, ty = backtest_profit_threshold_with_trades(
            prices, preds, INITIAL_CAPITAL, params["threshold"], COST_PCT,
            params["hold_hours"])
    elif strategy == "risk_based":
        pv, tt, tv, ty = backtest_profit_threshold(
            prices, preds, INITIAL_CAPITAL, params["threshold"],
            target_risk_per_trade=params["target_risk"],
            transaction_cost_pct=COST_PCT, hold_hours=params["hold_hours"])
    else:
        _, pv, tt, tv, ty = backtest_ts_momentum(
            prices=prices, predicted_changes=preds,
            lookback=params["lookback"], hold_hours=params["hold_hours"],
            target_risk_per_trade=params["target_risk"],
            threshold=params["signal_thresh"],
            initial_capital=INITIAL_CAPITAL, transaction_cost_pct=COST_PCT)
    return pv, tt, tv, ty


summary_rows = []
for strategy in ("threshold", "risk_based", "momentum"):
    for name in L.FEATURE_SET_ORDER:
        for r, info in REGIMES.items():
            sig = SIGNALS[(name, r)]
            prices = np.asarray(sig["close"], float)
            preds = np.asarray(sig["pred"], float)
            params = PARAMS[(name, strategy)]

            pv, tt, tv, ty = run_backtest(strategy, prices, preds, params)
            trades_df = L.create_trades_dataframe(
                tt, tv, ty, sig["idx"],
                f"{STRATEGY_LABEL[strategy]} {info['short']} — {L.NICE_NAME[name]}")

            folder = L.BT_DIR / L.FOLDER_NAME[name]
            folder.mkdir(parents=True, exist_ok=True)
            path = folder / f"{FILE_PREFIX[strategy]}-{r}.csv"
            trades_df.to_csv(path, index=False)

            summary_rows.append(dict(
                feature_set=name, strategy=STRATEGY_LABEL[strategy],
                regime=info["short"], trade_rows=len(trades_df),
                entries=int(trades_df["trade_type"].isin(["long", "short"]).sum()),
                final_value=float(pv[-1]),
                returns_pct=100 * (pv[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL,
                sharpe_annualised=L.calculate_sharpe_ratio(pv),
                max_dd_pct=abs(L.calculate_max_drawdown(pv)) * 100))
            print(f"{STRATEGY_LABEL[strategy]:11s} {info['short']:9s} "
                  f"{L.NICE_NAME[name]:28s} entries={summary_rows[-1]['entries']:4d}  "
                  f"final=${pv[-1]:9,.0f}  Sharpe={summary_rows[-1]['sharpe_annualised']:+.3f}  "
                  f"maxDD={summary_rows[-1]['max_dd_pct']:5.2f}%  -> {path.name}")

            plot_trades(trades_df,
                        f"{L.NICE_NAME[name]} — {STRATEGY_LABEL[strategy]} "
                        f"({info['short']}): ${pv[-1]:,.0f}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(L.OUT / "itransformer_backtest_summary_full_path.csv", index=False)
print()
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# ------------------------------------------------ model-free momentum benchmarks
bench_rows = []
for _key, _mode in BENCHMARKS.items():
    _params = PARAMS[(_key, "momentum")]
    for r, info in REGIMES.items():
        sig = SIGNALS[("price+volume", r)]
        prices = np.asarray(sig["close"], float)
        if _mode == "zero":
            preds = np.zeros(len(prices), float)
        else:
            preds = np.random.default_rng(0).permutation(np.asarray(sig["pred"], float))

        pv, tt, tv, ty = run_backtest("momentum", prices, preds, _params)
        trades_df = L.create_trades_dataframe(
            tt, tv, ty, sig["idx"], f"Momentum {info['short']} — {BENCH_LABEL[_key]}")
        folder = L.BT_DIR / _key
        folder.mkdir(parents=True, exist_ok=True)
        trades_df.to_csv(folder / f"{FILE_PREFIX['momentum']}-{r}.csv", index=False)

        bench_rows.append(dict(
            feature_set=BENCH_LABEL[_key], strategy="Momentum", regime=info["short"],
            trade_rows=len(trades_df),
            entries=int(trades_df["trade_type"].isin(["long", "short"]).sum()),
            final_value=float(pv[-1]),
            returns_pct=100 * (pv[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL,
            sharpe_annualised=L.calculate_sharpe_ratio(pv),
            max_dd_pct=abs(L.calculate_max_drawdown(pv)) * 100))
        print(f"{BENCH_LABEL[_key]:42s} {info['short']:9s} "
              f"entries={bench_rows[-1]['entries']:4d}  final=${pv[-1]:9,.0f}  "
              f"Sharpe={bench_rows[-1]['sharpe_annualised']:+.3f}")

bench = pd.DataFrame(bench_rows)
bench.to_csv(L.OUT / "itransformer_no_model_benchmark.csv", index=False)

# side-by-side: does the forecast add anything to the momentum rule?
mom = summary[summary["strategy"] == "Momentum"].copy()
mom["feature_set"] = mom["feature_set"].map(L.NICE_NAME)
cmp_tbl = (pd.concat([mom, bench])[["regime", "feature_set", "entries",
                                    "final_value", "returns_pct", "sharpe_annualised",
                                    "max_dd_pct"]]
             .sort_values(["regime", "final_value"], ascending=[True, False]))
print()
print("Momentum strategy: model-driven rows vs the same rule with no forecast")
print(cmp_tbl.to_string(index=False, float_format=lambda v: f"{v:.3f}"))


## 5. Strategy hyperparameters — LaTeX

Counterpart of the iTransformer strategy-parameter table.

In [ ]:
def _row(name, cells):
    return f"\t\t\t\t{L.NICE_NAME[name]} & " + " & ".join(cells) + r" \\"


lines = [r"\begin{table}[htbp]", "\t" + r"\centering",
         "\t" + r"\renewcommand{\arraystretch}{1.1}", "\t" + r"\scriptsize",
         "\t" + r"\begin{minipage}[t]{0.48\textwidth}", "\t\t" + r"\centering",
         "\t\t\t" + r"\begin{tabular}{|p{3.4cm}|p{1.8cm}|p{1.5cm}|}", "\t\t\t" + r"\hline",
         "\t\t\t\t" + r"Dataset & \textbf{Threshold ($\tau$)} & \textbf{Hold ($H$)} \\",
         "\t\t\t" + r"\hline"]
for name in L.FEATURE_SET_ORDER:
    p = PARAMS[(name, "threshold")]
    lines += [_row(name, [rf"$\mathbf{{{p['threshold']:.2f}\%}}$",
                          rf"$\mathbf{{{p['hold_hours']}}}$ h"]), "\t\t\t" + r"\hline"]
lines += ["\t\t\t" + r"\end{tabular}", "\t\t" + r"\subcaption{Threshold strategy}",
          "\t" + r"\end{minipage}", "\t" + r"\hfill",
          "\t" + r"\begin{minipage}[t]{0.48\textwidth}", "\t\t" + r"\centering",
          "\t\t\t" + r"\begin{tabular}{|p{3.0cm}|p{1.5cm}|p{1.3cm}|p{1.1cm}|}",
          "\t\t\t" + r"\hline",
          "\t\t\t\t" + r"Dataset & \textbf{Threshold ($\tau$)} & \textbf{Target Risk} & \textbf{Hold ($H$)} \\",
          "\t\t\t" + r"\hline"]
for name in L.FEATURE_SET_ORDER:
    p = PARAMS[(name, "risk_based")]
    lines += [_row(name, [rf"$\mathbf{{{p['threshold']:.2f}\%}}$",
                          rf"$\mathbf{{{p['target_risk']:.2f}}}$",
                          rf"$\mathbf{{{p['hold_hours']}}}$ h"]), "\t\t\t" + r"\hline"]
lines += ["\t\t\t" + r"\end{tabular}",
          "\t\t" + r"\subcaption{Risk-based (volatility-adjusted)}",
          "\t" + r"\end{minipage}", "\t" + r"\par\vspace{0.6em}",
          "\t" + r"\begin{minipage}[t]{0.7\textwidth}", "\t\t" + r"\centering",
          "\t\t\t" + r"\begin{tabular}{|p{3.4cm}|p{1.5cm}|p{1.6cm}|p{1.4cm}|p{1.2cm}|}",
          "\t\t\t" + r"\hline",
          "\t\t\t\t" + r"Dataset & \textbf{Lookback ($L$)} & \textbf{Threshold ($\tau$)} & \textbf{Target Risk} & \textbf{Hold ($H$)} \\",
          "\t\t\t" + r"\hline"]
for name in L.FEATURE_SET_ORDER:
    p = PARAMS[(name, "momentum")]
    lines += [_row(name, [rf"$\mathbf{{{p['lookback']}}}$",
                          rf"$\mathbf{{{p['signal_thresh']:.3f}\%}}$",
                          rf"$\mathbf{{{p['target_risk']:.3f}}}$",
                          rf"$\mathbf{{{p['hold_hours']}}}$ h"]), "\t\t\t" + r"\hline"]
lines += ["\t\t\t" + r"\end{tabular}", "\t\t" + r"\subcaption{Momentum}",
          "\t" + r"\end{minipage}",
          "\t" + r"\caption{Optimized iTransformer trading hyperparameters via CV+Optuna. "
                 r"Values shown per dataset and strategy.}",
          "\t" + r"\label{tab:itransformer_hyperparams_summary}", r"\end{table}"]
tex_params = "\n".join(lines)
(L.OUT / "table_itransformer_strategy_params.tex").write_text(tex_params)
param_tbl.to_csv(L.OUT / "itransformer_strategy_params.csv", index=False)
print(tex_params)

## 6. Done

In [ ]:
print("trade logs written under", L.BT_DIR)
for folder in sorted(L.BT_DIR.iterdir()):
    if folder.is_dir():
        print(" ", folder.name, "->", ", ".join(sorted(p.name for p in folder.iterdir())))
print("\nNext: run 03_itransformer_diagrams.ipynb for the comparison figures and the "
      "backtesting results table.")